Retail SQL Assistant

In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

d:\Resume Projects\retail_sqldb_assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [4]:
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    temperature = 0.2
)

In [5]:
poem = llm.invoke("Write a poem on my love for biryani")
print(poem)

content="When hunger calls, a whisper soft and low,\nOne name alone, my yearning heart does know.\nNo other dish can stir my soul so deep,\nWhile other cravings quietly sleep.\n\nFrom kitchen's depths, a fragrant cloud ascends,\nA symphony of spice, its magic sends.\nCardamom's kiss, and cinnamon's warm embrace,\nA promise whispered, filling every space.\n\nThe golden grains, like jewels, so bright they gleam,\nA saffron sunset, a culinary dream.\nThe tender meat, beneath its fragrant shroud,\nA hidden treasure, waiting to be found.\n\nEach fluffy grain, a story to unfold,\nOf spices dancing, vibrant, bold, and old.\nThe succulent bite, the layers deep and true,\nA flavour journey, fresh and ever new.\n\nThe chicken tender, melting on the tongue,\nOr mutton rich, a flavour anthem, beautifully sung.\nWith crispy onions, a delightful crunch,\nA perfect balance, from the very first munch.\n\nIt's more than food, it's comfort, warmth, and grace,\nA happy memory, time cannot erase.\nMy hear

Connect to the Local My SQL Database workbench

In [ ]:
from langchain_community.utilities import SQLDatabase
db_user = "root"
db_password = "your_password"
db_host = "localhost"
db_name="atliq_tshirts"

database_uri = f"mysql+mysqlconnector://{db_user}:{db_password}@{db_host}/{db_name}"
db = SQLDatabase.from_uri(database_uri=database_uri, sample_rows_in_table_info = 3)
print(db.table_info)


CREATE TABLE discounts (
	discount_id INTEGER NOT NULL AUTO_INCREMENT, 
	t_shirt_id INTEGER NOT NULL, 
	pct_discount DECIMAL(5, 2), 
	PRIMARY KEY (discount_id), 
	CONSTRAINT discounts_ibfk_1 FOREIGN KEY(t_shirt_id) REFERENCES t_shirts (t_shirt_id), 
	CONSTRAINT discounts_chk_1 CHECK ((`pct_discount` between 0 and 100))
)DEFAULT CHARSET=utf8mb4 ENGINE=InnoDB COLLATE utf8mb4_0900_ai_ci

/*
3 rows from discounts table:
discount_id	t_shirt_id	pct_discount
1	1	10.00
2	2	15.00
3	3	20.00
*/


CREATE TABLE t_shirts (
	t_shirt_id INTEGER NOT NULL AUTO_INCREMENT, 
	brand ENUM('Van Huesen','Levi','Nike','Adidas') NOT NULL, 
	color ENUM('Red','Blue','Black','White') NOT NULL, 
	size ENUM('XS','S','M','L','XL') NOT NULL, 
	price INTEGER, 
	stock_quantity INTEGER NOT NULL, 
	PRIMARY KEY (t_shirt_id), 
	CONSTRAINT t_shirts_chk_1 CHECK ((`price` between 10 and 50))
)DEFAULT CHARSET=utf8mb4 ENGINE=InnoDB COLLATE utf8mb4_0900_ai_ci

/*
3 rows from t_shirts table:
t_shirt_id	brand	color	size	price	stock

In [7]:
from langchain_experimental.sql import SQLDatabaseChain

db_chain = SQLDatabaseChain.from_llm(llm = llm, db = db)

In [12]:
qns1 = db_chain.invoke("How many extra small size white color nike t-shirts are left ?")

In [13]:
qns1

{'query': 'How many extra small size white color nike t-shirts are left ?',
 'result': 'Answer: There are 53 extra small size white color Nike t-shirts left.'}

In [20]:
qns1 = db_chain.invoke("SELECT `stock_quantity` FROM t_shirts WHERE `brand` = 'Nike' AND `size` = 'XS' AND `color` = 'White'")

In [21]:
qns1

{'query': "SELECT `stock_quantity` FROM t_shirts WHERE `brand` = 'Nike' AND `size` = 'XS' AND `color` = 'White'",
 'result': 'Answer: The stock quantity for Nike, XS, White t-shirts is 53.'}

In [22]:
# To return only the answer use the "invoke" method from chain
qns2 = db_chain.invoke("How much is the price of the inventory for all small size t-shirts?")

In [23]:
qns2

{'query': 'How much is the price of the inventory for all small size t-shirts?',
 'result': 'Answer: The total price of the inventory for all small size t-shirts is 21998.'}

In [24]:
# Also run the actual query with the db_chain
qns2 = db_chain.invoke("SELECT SUM(price * stock_quantity) FROM t_shirts WHERE size = 'S'")

In [25]:
qns2

{'query': "SELECT SUM(price * stock_quantity) FROM t_shirts WHERE size = 'S'",
 'result': '21998'}

In [26]:
qns3 = db_chain.invoke("If we have to sell all the Levi's T-shirts today with discounts applied. How much revenue our store will generate (post discounts)?")

In [27]:
qns3

{'query': "If we have to sell all the Levi's T-shirts today with discounts applied. How much revenue our store will generate (post discounts)?",
 'result': "Answer: Our store will generate 4488.30 in revenue post discounts from selling all Levi's T-shirts today."}

In [28]:
sql_query="""
select sum(a.total_amount * ((100 - COALESCE(discounts.pct_discount,0))/100)) as total_revenue from
(select sum(price*stock_quantity) as total_amount, t_shirt_id from t_shirts where brand = 'Levi'
group by t_shirt_id) a left join discounts on a.t_shirt_id = discounts.t_shirt_id
"""
qns3 = db_chain.invoke(sql_query)

In [29]:
qns3

{'query': "\nselect sum(a.total_amount * ((100 - COALESCE(discounts.pct_discount,0))/100)) as total_revenue from\n(select sum(price*stock_quantity) as total_amount, t_shirt_id from t_shirts where brand = 'Levi'\ngroup by t_shirt_id) a left join discounts on a.t_shirt_id = discounts.t_shirt_id\n",
 'result': 'The total revenue from Levi t-shirts, considering any applicable discounts, is 31526.30.'}

In [31]:
qns4 = db_chain.invoke("SELECT SUM(price * stock_quantity) FROM t_shirts WHERE brand = 'Levi'")

In [32]:
qns4

{'query': "SELECT SUM(price * stock_quantity) FROM t_shirts WHERE brand = 'Levi'",
 'result': 'The total value of Levi t-shirts in stock is 33351.'}

In [70]:
qns5 = db_chain.invoke("How many Levi's white color t-shirts of are available?")

In [71]:
qns5

{'query': "How many Levi's white color t-shirts of are available?",
 'result': "There are 203 Levi's white color t-shirts available."}

In [80]:
qns5 = db_chain.invoke("SELECT SUM(`stock_quantity`) FROM `t_shirts` WHERE `color` = 'White' AND `brand` = 'Levi'")

In [81]:
qns5

{'query': "SELECT SUM(`stock_quantity`) FROM `t_shirts` WHERE `color` = 'White' AND `brand` = 'Levi'",
 'result': "SQLQuery: SELECT SUM(`stock_quantity`) FROM `t_shirts` WHERE `color` = 'White' AND `brand` = 'Levi'"}

Few Shot Learning

In [82]:
few_shots = [
    {'Question' : "How many t-shirts do we have left for Nike in XS size and white color?",
     'SQLQuery' : "SELECT sum(stock_quantity) FROM t_shirts WHERE brand = 'Nike' AND color = 'White' AND size = 'XS'",
     'SQLResult': "Result of the SQL query",
     'Answer' : "The number of white color extra small size t-shirts of nike left are 53"},
    {'Question': "How much is the total price of the inventory for all S-size t-shirts?",
     'SQLQuery':"SELECT SUM(price*stock_quantity) FROM t_shirts WHERE size = 'S'",
     'SQLResult': "Result of the SQL query",
     'Answer': qns2['result']},
    {'Question': "If we have to sell all the Levi’s T-shirts today with discounts applied. How much revenue  our store will generate (post discounts)?" ,
     'SQLQuery' : """select sum(a.total_amount * ((100 - COALESCE(discounts.pct_discount,0))/100)) as total_revenue from
                    (select sum(price*stock_quantity) as total_amount, t_shirt_id from t_shirts where brand = 'Levi'
                    group by t_shirt_id) a left join discounts on a.t_shirt_id = discounts.t_shirt_id
        """,
     'SQLResult': "Result of the SQL query",
     'Answer': qns3['result']} ,
     {'Question' : "If we have to sell all the Levi’s T-shirts today. How much revenue our store will generate without discount?" ,
      'SQLQuery': "SELECT SUM(price * stock_quantity) FROM t_shirts WHERE brand = 'Levi'",
      'SQLResult': "Result of the SQL query",
      'Answer' : qns4['result']},
    {'Question': "How many white color Levi's shirt I have?",
     'SQLQuery' : "SELECT sum(stock_quantity) FROM t_shirts WHERE brand = 'Levi' AND color = 'White'",
     'SQLResult': "Result of the SQL query",
     'Answer' : qns5['result']
     }
]

In [83]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-mpnet-base-v2")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [84]:
e = embeddings.embed_query("How many white color Levi's shirt I have?")
len(e), type(e)

(768, list)

In [85]:
e[:5]

[-0.012699399143457413,
 0.04724609851837158,
 0.0040471819229424,
 0.05207887664437294,
 0.0002692432899493724]

In [86]:
to_vectorize = [" ".join(example.values()) for example in few_shots]
to_vectorize[1]

"How much is the total price of the inventory for all S-size t-shirts? SELECT SUM(price*stock_quantity) FROM t_shirts WHERE size = 'S' Result of the SQL query 21998"

In [87]:
from langchain_chroma import Chroma

vector_store = Chroma.from_texts(
    texts = to_vectorize,
    embedding = embeddings,
    metadatas = few_shots
)

In [88]:
from langchain_core.example_selectors.semantic_similarity import SemanticSimilarityExampleSelector

example_selector = SemanticSimilarityExampleSelector(
    vectorstore=vector_store,
    k = 2
)

example_selector.select_examples({"Question": "How many Adidas T shirts I have left in my store?"})

[{'Question': 'How many t-shirts do we have left for Nike in XS size and white color?',
  'SQLResult': 'Result of the SQL query',
  'SQLQuery': "SELECT sum(stock_quantity) FROM t_shirts WHERE brand = 'Nike' AND color = 'White' AND size = 'XS'",
  'Answer': 'The number of white color extra small size t-shirts of nike left are 53'},
 {'SQLQuery': "SELECT SUM(price * stock_quantity) FROM t_shirts WHERE brand = 'Levi'",
  'SQLResult': 'Result of the SQL query',
  'Question': 'If we have to sell all the Levi’s T-shirts today. How much revenue our store will generate without discount?',
  'Answer': 'The total value of Levi t-shirts in stock is 33351.'}]

In [89]:
from langchain.chains.sql_database.prompt import PROMPT_SUFFIX, _mysql_prompt

In [90]:
print(_mysql_prompt)

You are a MySQL expert. Given an input question, first create a syntactically correct MySQL query to run, then look at the results of the query and return the answer to the input question.
Unless the user specifies in the question a specific number of examples to obtain, query for at most {top_k} results using the LIMIT clause as per MySQL. You can order the results to return the most informative data in the database.
Never query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in backticks (`) to denote them as delimited identifiers.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.
Pay attention to use CURDATE() function to get the current date, if the question involves "today".

Use the following format:

Question: Question here
SQLQuery: SQL Query to run
SQLResult: Result of

In [91]:
print(PROMPT_SUFFIX)

Only use the following tables:
{table_info}

Question: {input}


In [93]:
from langchain_core.prompts import PromptTemplate

template = "Question: {Question} \n SQLQuery: {SQLQuery} \n SQLResult: {SQLResult}\n Answer: {Answer}"
example_prompt = PromptTemplate.from_template(template)

In [94]:
from langchain_core.prompts import FewShotPromptTemplate
fewshot_template = FewShotPromptTemplate(
    example_selector = example_selector,
    example_prompt = example_prompt,
    input_variables = ["input", "table_info", "top_k"],
    suffix = PROMPT_SUFFIX
)

In [95]:
new_chain = SQLDatabaseChain.from_llm(llm = llm, db=db, prompt=fewshot_template)

In [96]:
new_chain.invoke("How many white color Levi's t-shirts are available?")

{'query': "How many white color Levi's t-shirts are available?",
 'result': "The number of white color Levi's t-shirts available are 203"}

In [97]:
new_chain.invoke("How much is the price of the inventory for all small size t-shirts?")

{'query': 'How much is the price of the inventory for all small size t-shirts?',
 'result': '21998'}

In [98]:
new_chain.invoke("How much is the price of all white color levi t shirts?")

{'query': 'How much is the price of all white color levi t shirts?',
 'result': '5659'}

In [99]:
new_chain.invoke("How much revenue our store will generate by selling all Van Heuson TShirts without discount?")

{'query': 'How much revenue our store will generate by selling all Van Heuson TShirts without discount?',
 'result': 'The total revenue generated by selling all Van Heusen T-shirts without discount is 42545.'}